# 2.2 · LGD fine-tuning

How every Exp2 fine-tuning configuration behaved *while it trained* — loss, real-data R-squared, out-of-domain retention, all logged metrics, hardware and gradient flow. Reads the per-arm progress and telemetry manifests, so it works on a partial sweep.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src.visualize import training_plots, figures, style

style.apply()   # ONE shared style: identical colours in every notebook
pd.set_option("display.width", 200, "display.max_columns", 40)

TASK = "lgd"
EXP = "exp2"
# Exp2 fine-tunes the released TabICLv2 weights on our credit prior. Every started arm leaves
# output/manifests/exp2_{TASK}__<run>__progress.csv and __telemetry.csv; these read them, so a
# partial sweep still plots. Clears THIS notebook's figure folder before drawing.
FIGS = figures.FigureSaver("2.2_lgd_finetuning")

## 1. Training loss

In [ ]:
FIGS.save(training_plots.training_loss(TASK, exp=EXP), "training_loss",
    caption="Training loss against optimisation step for every LGD fine-tuning arm (grey), the mean across arms (black), and the best and worst arms by final real-data R-squared (highlighted).");

## 2. Real-data R-squared over training

In [ ]:
FIGS.save(training_plots.metric_over_training(TASK, exp=EXP), "metric_over_training",
    caption="Real-data R-squared, averaged over the evaluation datasets, against training step; one line per arm coloured by prior (credit versus continued-pretraining control), with the across-arm mean in black.");

## 3. Credit versus out-of-domain retention

In [ ]:
FIGS.save(training_plots.real_vs_ood(TASK, exp=EXP), "real_vs_ood",
    caption="Mean R-squared on the real-credit datasets (solid) and the out-of-domain suites (dashed) against training step, averaged across fine-tuning arms.");

## 4. Every logged evaluation metric

In [ ]:
for _p in range(1, training_plots.metric_pages(TASK, exp=EXP) + 1):
    FIGS.save(training_plots.all_eval_metrics(TASK, _p, exp=EXP), f"eval_metrics_p{_p}",
        caption="Each logged real-data evaluation metric, averaged over arms and datasets, against training step; the arrow in each panel title marks the improving direction.");

## 5. Per-configuration training curves

In [ ]:
for _p in range(1, training_plots.config_pages(TASK, exp=EXP) + 1):
    FIGS.save(training_plots.per_config(TASK, _p, exp=EXP), f"per_config_p{_p}",
        caption="Per-arm training curves against step: train loss (grey, left axis) and real-data R-squared (blue, right axis), one panel per fine-tuning configuration.");

## 6. Best versus worst configuration

In [ ]:
FIGS.save(training_plots.best_and_worst(TASK, exp=EXP), "best_and_worst",
    caption="Train loss and real-data R-squared against training step for the best and worst fine-tuning arm by final R-squared.");

## 7. Hardware during training

In [ ]:
FIGS.save(training_plots.hardware(TASK, exp=EXP), "hardware",
    caption="GPU utilisation, throughput and peak allocated memory against training step, pooled across arms; the dashed line marks 70 percent utilisation.");

## 8. Per-block gradient flow

In [ ]:
FIGS.save(training_plots.gradient_flow(TASK, exp=EXP), "gradient_flow",
    caption="Mean per-block gradient L2 norm (column encoder, row encoder, ICL blocks, head) against training step on a logarithmic axis; a frozen stack sits on the floor.");

## Summary

In [ ]:
print(training_plots.training_summary(TASK, exp=EXP))
print()
print(FIGS.summary())